# Algebra Lab

Infinite practice for **factoring**, **systems of equations**, and **linear algebra** — with worked steps and automatic grading.

Everything here is generated on the fly, so you never run out of problems and never see the same one twice.

| topic | what you get |
|---|---|
| `factoring` | GCF, difference of squares, perfect squares, trinomials, grouping |
| `systems` | 2x2 and 3x3, including *no solution* and *infinitely many* |
| `gauss` | row reduction, read the solution off the RREF |
| `eigen` | eigenvalues and eigenvectors (always real integers) |
| `det` | 3x3 determinants by cofactor expansion |

**How to type answers**

- `^` works for powers, so `x^2` is fine
- `*` is optional: `5x^2(6x+4)` and `(2x - 3)(5x - 3)` are both understood
- systems: `x=2, y=-1`
- eigenvalues: `lam=3, lam=-1, v=[1,2]`
- for no/infinite solutions just type `no solution` or `infinitely many`

While answering you can type `hint`, `skip`, or `answer` to bail out gracefully.

In [ ]:
import random
import algebra_lab as lab

TOPICS = {
    "factoring": lab.FACTORING,
    "systems": [lambda: lab.system("2x2"), lambda: lab.system("3x3")],
    "gauss": [lab.gauss_jordan],
    "eigen": [lab.eigenvalues_2x2],
    "det": [lab.determinant_3x3],
}
TOPICS["random"] = lab.FACTORING + lab.LINEAR_ALGEBRA

def seed(n=None):
    """Fix the random seed so a problem set can be reproduced."""
    random.seed(n)
    print("seed =", random.getstate()[1][0] if n is None else n)

print("ready — try practice('factoring')")

## `practice(topic)` — graded, one problem at a time

The main event. Loops until you stop.

In [ ]:
def practice(topic="random", show_steps_always=False):
    """Interactive drill. `topic` is a key of TOPICS, or 'random'."""
    pool = TOPICS.get(topic, TOPICS["random"])
    right = wrong = 0
    streak = 0
    best = 0
    while True:
        p = random.choice(pool)()
        print("=" * 66)
        print(p["kind"])
        print("-" * 66)
        print(p["problem"])
        print()
        try:
            raw = input("your answer> ").strip()
        except (EOFError, KeyboardInterrupt):
            print()
            break
        low = raw.lower()
        if low in {"q", "quit", "exit"}:
            break
        if low in {"h", "hint"}:
            print("  hint:", p["hint"], "\n")
            continue
        if low in {"a", "ans", "answer", "skip", "s"}:
            reveal(p)
            wrong += 1
            streak = 0
        else:
            ok, msg = p["check"](raw)
            print(f"  {'CORRECT' if ok else 'not yet'}  {msg}\n")
            if ok:
                right += 1
                streak += 1
                best = max(best, streak)
            else:
                wrong += 1
                streak = 0
                reveal(p)
        print(f"  score {right} right / {wrong} not yet   streak {streak} (best {best})\n")
        try:
            if input("another? (y/n) ").strip().lower().startswith("n"):
                break
        except (EOFError, KeyboardInterrupt):
            print()
            break
    print(f"\nfinal: {right} right, {wrong} not yet, best streak {best}")

def reveal(p):
    """Print the answer and the worked steps."""
    print(f"\n  ANSWER:  {p['answer']}")
    print("  STEPS:")
    for s in p["steps"]:
        for line in str(s).splitlines():
            print("    *", line)
    print()

print("defined. e.g.  practice('factoring')")

## `drill(topic, n)` — no typing, just problems + answers

Use this to see the *kinds* of thing that come up, or to self-test on paper first.

In [ ]:
def drill(topic="random", n=5, with_steps=True):
    """Print n problems with answers. No input required."""
    pool = TOPICS.get(topic, TOPICS["random"])
    for _ in range(n):
        p = random.choice(pool)()
        print("=" * 66)
        print(p["kind"])
        print("-" * 66)
        print(p["problem"])
        print(f"\n  ANSWER:  {p['answer']}")
        if with_steps:
            print("  STEPS:")
            for s in p["steps"]:
                for line in str(s).splitlines():
                    print("    *", line)
        print()

drill('factoring', 3)

## `check(topic, n)` — grade yourself silently

Generates n problems and runs each one's own grader against its own answer, as a smoke test that the generator is healthy. Useful after you change something.

In [ ]:
def selftest(topic="random", n=200):
    """Verify every generator: answers must grade correct, junk must not."""
    pool = TOPICS.get(topic, TOPICS["random"])
    junk = ["", "nonsense", "zzz", "(", "1/0", "???", "x=??", "asdf"]
    bad = 0
    for _ in range(n):
        p = random.choice(pool)()
        ok, msg = p["check"](p["answer"])
        if not ok:
            print("SELF-FAIL", p["kind"], repr(p["answer"]), msg)
            bad += 1
        for j in junk:
            if p["check"](j)[0]:
                print("FALSE-ACCEPT", p["kind"], repr(j))
                bad += 1
    print(f"{n} problems checked, {bad} problems")
    return bad == 0

selftest('random', 100)

## `menu()`

Pick a topic from a list.

In [ ]:
def menu():
    labels = {
        "1": ("factoring", "factoring — GCF, squares, trinomials, grouping"),
        "2": ("systems", "systems of equations — 2x2 and 3x3"),
        "3": ("gauss", "row reduction (Gauss-Jordan)"),
        "4": ("eigen", "eigenvalues and eigenvectors"),
        "5": ("det", "3x3 determinants"),
        "r": ("random", "random — everything"),
    }
    while True:
        for k, (_, text) in labels.items():
            print(f"  {k}  {text}")
        c = input("choose (or q to quit)> ").strip().lower()
        if c in {"q", "quit", "exit"}:
            return
        if c not in labels:
            print("  pick one of the letters above")
            continue
        key, _ = labels[c]
        practice(key)

print("defined. run  menu()")

## Same thing from the terminal

The logic lives in `algebra_lab.py`, which is also a CLI:

```bash
# see 5 problems with answers
.venv/bin/python algebra_lab.py factoring -n 5

# graded drill, stops when you say stop
.venv/bin/python algebra_lab.py systems --practice

# a repeatable set
.venv/bin/python algebra_lab.py random -n 10 -s 42
```